# Skipped Steps Explainer
## What the pipeline skips on tutorial data — and what it would look like in production

This notebook explains every step that is skipped in the demo pipeline due to hardware constraints,
showing representative outputs, expected metric ranges, and the full production commands.

**Skipped steps covered:**
1. Read QC (FastQC / MultiQC)
2. Adapter Trimming (fastp / Trimmomatic)
3. BWA-MEM Alignment (FASTQ → BAM)
4. BQSR (Base Quality Score Recalibration)
5. VQSR (Variant Quality Score Recalibration)

Each section follows: **Concept → What it measures → Expected metrics → Red flags → Production command → Why skipped here**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size']  = 11
print('Libraries loaded.')

---
## Step 1: Read QC — FastQC / MultiQC

### Concept
Before any alignment or variant calling, raw FASTQ files from the sequencer are quality-checked.
FastQC generates a detailed HTML report for each FASTQ file. MultiQC aggregates reports from
multiple samples into a single summary.

### What FastQC measures
| Module | What it checks | Good range |
|--------|---------------|------------|
| Per-base sequence quality | Phred Q-score at each read position | Q30+ across all positions |
| Per-sequence quality scores | Distribution of mean read quality | Peak at Q35+ |
| Per-base sequence content | % of A/T/G/C at each position | ~25% each |
| GC content | Overall GC% vs. expected | Species-specific (~50% human) |
| Sequence length distribution | Consistency of read lengths | Uniform (e.g. all 150bp) |
| Sequence duplication levels | % duplicate reads | < 20% for WGS |
| Adapter content | Presence of sequencing adapters | < 1% |
| Overrepresented sequences | Any sequence appearing > 0.1% | None, or known adapters |

### Why skipped
The tutorial BAMs are pre-processed and come from a curated dataset — no raw FASTQ available.
In production, FastQC is always the FIRST step before any processing.

In [ ]:
# Simulate a representative FastQC per-base quality plot
# This is what a good NovaSeq 150bp paired-end run looks like
np.random.seed(42)
positions = np.arange(1, 151)

# Read 1: slight quality drop at the end (normal for Illumina)
q_mean_r1   = 37 - 0.03 * positions + np.random.normal(0, 0.3, 150)
q_q10_r1    = q_mean_r1 - 4 + np.random.normal(0, 0.5, 150)
q_q90_r1    = q_mean_r1 + 4 + np.random.normal(0, 0.5, 150)
q_mean_r1   = np.clip(q_mean_r1, 20, 40)
q_q10_r1    = np.clip(q_q10_r1, 15, 40)
q_q90_r1    = np.clip(q_q90_r1, 25, 40)

# Read 2: typically slightly lower quality
q_mean_r2   = 36 - 0.04 * positions + np.random.normal(0, 0.3, 150)
q_mean_r2   = np.clip(q_mean_r2, 18, 40)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Per-base quality (Read 1)
ax = axes[0]
ax.fill_between(positions, q_q10_r1, q_q90_r1, alpha=0.2, color='green', label='Q10–Q90 range')
ax.plot(positions, q_mean_r1, color='darkblue', linewidth=2, label='Mean quality')
ax.axhline(30, color='orange', linestyle='--', linewidth=1.2, label='Q30 threshold')
ax.axhline(20, color='red',    linestyle='--', linewidth=1.2, label='Q20 (poor)')
ax.fill_between(positions, 0, 20,  alpha=0.05, color='red')
ax.fill_between(positions, 20, 28, alpha=0.05, color='orange')
ax.fill_between(positions, 28, 40, alpha=0.05, color='green')
ax.set_ylim(0, 42)
ax.set_xlabel('Position in Read (bp)')
ax.set_ylabel('Phred Quality Score')
ax.set_title('FastQC: Per-Base Sequence Quality (Read 1)\n[Representative — NovaSeq 150bp PE]')
ax.legend(fontsize=9)
ax.text(130, 38, 'Q30 = 99.9%\naccuracy', fontsize=8, color='green')

# Right: Per-sequence quality distribution
ax = axes[1]
quality_scores = np.random.normal(37, 1.5, 100000).clip(20, 40)
ax.hist(quality_scores, bins=40, color='steelblue', edgecolor='white', alpha=0.85, density=True)
ax.axvline(30, color='orange', linestyle='--', linewidth=1.5, label='Q30')
ax.set_xlabel('Mean Sequence Quality Score')
ax.set_ylabel('Density')
ax.set_title('FastQC: Per-Sequence Quality Scores\n[Good run: peak at Q35+]')
ax.legend()

plt.tight_layout()
plt.savefig('../results/docs/fastqc_representative.png', bbox_inches='tight')
plt.show()

print('INTERPRETATION:')
print('  Green zone (Q28+): high quality — keep all reads')
print('  Orange zone (Q20-28): moderate — trimming may help')
print('  Red zone (<Q20): poor quality — trim or discard')
print()
print('RED FLAGS to look for:')
print('  • Quality drop throughout the read (not just at end) → sequencing run problem')
print('  • Bimodal GC distribution → contamination from another species')
print('  • Adapter content > 5% → trimming required before alignment')
print('  • Per-base N content > 5% → base-calling failure zones')

In [ ]:
# Production command
print('=== PRODUCTION COMMAND ===')
print('''
# Run FastQC on all FASTQs
fastqc data/fastq/*.fastq.gz -o results/fastqc/ -t 8

# Aggregate with MultiQC
multiqc results/fastqc/ -o results/multiqc/

# Typical runtime: ~5 min per sample
# Memory: < 500 MB
# Output: HTML report per sample + aggregated MultiQC HTML
''')

---
## Step 2: Adapter Trimming — fastp

### Concept
Illumina sequencing appends adapter sequences to the ends of DNA fragments. If a fragment
is shorter than the read length (e.g. 100bp fragment, 150bp reads), the sequencer reads
into the adapter. These adapter bases do not come from the genome and must be removed.

**fastp** detects adapters automatically and performs quality trimming in a single pass.

### What trimming removes
- Adapter sequences (detected from known Illumina adapter sequences or auto-detected)
- Low-quality bases at read ends (Phred < 20 in a sliding window)
- Very short reads after trimming (< 30bp)
- Poly-G tails (common on NextSeq/NovaSeq where dark cycles produce G calls)

In [ ]:
# Simulate fastp trimming statistics
stats_before = {
    'Total reads':       '80,000,000',
    'Total bases':       '12.0 Gbp',
    'Q20 rate':          '97.2%',
    'Q30 rate':          '93.1%',
    'GC content':        '49.8%',
    'Adapter rate':      '8.3%',
    'Duplication rate':  '12.4%',
}
stats_after = {
    'Total reads':       '79,200,000 (99.0% retained)',
    'Total bases':       '11.5 Gbp',
    'Q20 rate':          '98.9%',
    'Q30 rate':          '95.8%',
    'GC content':        '49.9%',
    'Adapter rate':      '0.0% (removed)',
    'Duplication rate':  '12.4% (unchanged — trim does not remove dups)',
}

print('=== Representative fastp Summary ===')
print(f'{"Metric":<25} {"Before trimming":<25} {"After trimming"}')
print('-' * 80)
for k in stats_before:
    print(f'{k:<25} {stats_before[k]:<25} {stats_after.get(k, "-")}')

print()

# Show adapter content plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

positions = np.arange(1, 151)
adapter_before = 1 / (1 + np.exp(-0.05 * (positions - 100)))  # sigmoid at position 100
adapter_after  = np.zeros(150) + 0.0001

axes[0].plot(positions, adapter_before * 100, color='red',   linewidth=2, label='Before trimming')
axes[0].plot(positions, adapter_after  * 100, color='green', linewidth=2, label='After trimming')
axes[0].axhline(1, color='orange', linestyle='--', label='1% alert threshold')
axes[0].set_xlabel('Position in Read (bp)')
axes[0].set_ylabel('% Reads with Adapter')
axes[0].set_title('Adapter Content: Before vs. After Trimming')
axes[0].legend()

# Read length distribution after trimming
np.random.seed(0)
read_lengths = np.concatenate([
    np.random.normal(149, 1, 70000),    # full-length reads (most common)
    np.random.normal(120, 8, 15000),    # trimmed reads
    np.random.uniform(30, 100, 5000),   # heavily trimmed short inserts
]).clip(30, 151).astype(int)

axes[1].hist(read_lengths, bins=50, color='steelblue', edgecolor='white', alpha=0.85)
axes[1].set_xlabel('Read Length After Trimming (bp)')
axes[1].set_ylabel('Count')
axes[1].set_title('Read Length Distribution After fastp Trimming')
axes[1].text(40, axes[1].get_ylim()[1]*0.7,
             'Short reads\n(small inserts)', fontsize=9, ha='left')

plt.tight_layout()
plt.show()

print('\n=== PRODUCTION COMMAND ===')
print('''
fastp \
    -i sample_R1.fastq.gz -I sample_R2.fastq.gz \
    -o sample_R1_trimmed.fastq.gz -O sample_R2_trimmed.fastq.gz \
    --detect_adapter_for_pe \
    --cut_right --cut_right_mean_quality 20 \
    --length_required 30 \
    --json results/fastqc/sample_fastp.json \
    --html results/fastqc/sample_fastp.html \
    --thread 8

# Typical runtime: 10-20 min per 30x WGS sample
# Memory: < 300 MB
''')

---
## Step 3: BWA-MEM Alignment (FASTQ → BAM)

### Concept
Alignment maps each read in the FASTQ file to its best matching position in the reference genome.
BWA-MEM (Burrows-Wheeler Aligner, Maximal Exact Matches) is the standard tool for Illumina reads.

**How BWA-MEM works:**
1. Build a BWT (Burrows-Wheeler Transform) index of the reference genome (~3 GB for hg38)
2. For each read, find maximal exact matches (seeds) in the index
3. Extend seeds using Smith-Waterman local alignment to handle mismatches/gaps
4. Choose the best alignment location (mapq score reflects uniqueness)

**Read groups (RG) are critical:**
GATK requires RG tags in the BAM header to distinguish samples and sequencing units.
A missing or incorrect RG tag causes GATK to fail or produce wrong results.

### Expected `samtools flagstat` output for a good WGS alignment

In [ ]:
# Simulate samtools flagstat output
flagstat_good = """80000000 + 0 in total (QC-passed reads + QC-failed reads)
79600000 + 0 primary
     400000 + 0 supplementary  (chimeric reads, split alignments)
          0 + 0 duplicates     (run MarkDuplicates first)
79568000 + 0 primary mapped   (99.96%) ← GOOD: want > 99%
80000000 + 0 paired in sequencing
40000000 + 0 read1
40000000 + 0 read2
78902000 + 0 properly paired  (98.6%) ← GOOD: want > 95%
   650000 + 0 with itself and mate mapped
    50000 + 0 singletons       (0.06%) ← GOOD: want < 1%
   180000 + 0 with mate mapped to different chr (mapQ>=5)"""

flagstat_bad = """80000000 + 0 in total
79800000 + 0 primary
  5200000 + 0 primary mapped   (65.2%) ← BAD: < 95% suggests contamination or wrong reference
80000000 + 0 paired in sequencing
56000000 + 0 properly paired  (70.0%) ← BAD: suggests fragment size mismatch or library issues
  8000000 + 0 singletons       (10.0%) ← BAD: indicates alignment problems"""

print('=== Good alignment (samtools flagstat) ===')
print(flagstat_good)
print()
print('=== Problematic alignment (WARNING) ===')
print(flagstat_bad)

# Coverage histogram
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

np.random.seed(1)
depth_30x = np.random.negative_binomial(30, 0.5, 100000)
depth_30x = depth_30x[depth_30x < 120]

axes[0].hist(depth_30x, bins=80, color='#4472C4', edgecolor='white', alpha=0.85, density=True)
axes[0].axvline(20, color='orange', linestyle='--', linewidth=1.5, label='Min variant calling (20×)')
axes[0].axvline(30, color='green',  linestyle='--', linewidth=1.5, label='Target WGS depth (30×)')
axes[0].set_xlabel('Sequencing Depth (× coverage)')
axes[0].set_ylabel('Density')
axes[0].set_title('Coverage Distribution — 30× WGS\n[Representative Negative Binomial]')
axes[0].legend(fontsize=9)

# Mapping quality distribution
np.random.seed(2)
mapq_vals = np.concatenate([
    np.ones(5000) * 0,    # multi-mappers (assign mapQ=0)
    np.random.choice([1, 2, 3, 5], 2000),
    np.ones(93000) * 60,  # uniquely mapped (mapQ=60 for BWA-MEM)
])
axes[1].hist(mapq_vals, bins=62, range=(-0.5, 60.5), color='#ED7D31', edgecolor='white', alpha=0.85)
axes[1].set_xlabel('Mapping Quality (MAPQ)')
axes[1].set_ylabel('Count')
axes[1].set_title('Mapping Quality Distribution\n(MAPQ=60: uniquely mapped; MAPQ=0: multi-mapper)')
axes[1].text(1, axes[1].get_ylim()[1]*0.85, 'Multi-mappers\n(repetitive regions)', fontsize=9)
axes[1].text(50, axes[1].get_ylim()[1]*0.85, 'Unique\nmapping', fontsize=9)

plt.tight_layout()
plt.show()

print('\n=== PRODUCTION COMMAND ===')
print('''
# 1. Index reference (one-time, ~1 hour)
bwa index data/ref/Homo_sapiens_assembly38.fasta
samtools faidx data/ref/Homo_sapiens_assembly38.fasta

# 2. Align and sort
bwa mem -t 16 \
    -R "@RG\tID:tumor\tSM:HCC1143\tPL:ILLUMINA\tLB:lib1\tPU:flowcell1.lane1" \
    data/ref/Homo_sapiens_assembly38.fasta \
    data/fastq/tumor_R1_trimmed.fastq.gz \
    data/fastq/tumor_R2_trimmed.fastq.gz \
    | samtools sort -@ 8 -m 2G -o results/aligned/tumor.bam

samtools index results/aligned/tumor.bam
samtools flagstat results/aligned/tumor.bam > results/aligned/tumor.flagstat

# Typical runtime: 4 hours/sample at 30× WGS, 16 threads
# Memory: ~8 GB for hg38 index + 2 GB/sort thread
''')

---
## Step 4: BQSR — Base Quality Score Recalibration

### Concept
The sequencer reports a Phred quality score (Q) for each base call. A Q30 score means
the sequencer thinks there's a 1-in-1000 chance of error. But these scores are **miscalibrated**:
- Some contexts (e.g. GGT trinucleotide) have systematically higher error rates than reported
- Positions near the end of a read have lower quality than the sequencer admits
- PCR-introduced errors in specific contexts are not fully captured by cycle-based models

BQSR corrects these biases by:
1. Comparing observed error rates at **known variant sites** to expected rates
2. Building a model of: error rate = f(read group, reported quality, cycle, dinucleotide context)
3. Applying corrected quality scores to every base in the BAM

### Covariates modeled
| Covariate | Description | Example effect |
|-----------|-------------|---------------|
| Read group | Which sequencing run | Run-specific error profiles |
| Reported quality | Sequencer's own estimate | Starting point for correction |
| Cycle | Position within the read | End-of-read quality drop |
| Dinucleotide | Base preceding this base | GA context has higher G→T error |

### Why skipped
- Requires 16–32 GB RAM for chromosome-scale BAMs
- On modern NovaSeq (patterned flowcell), base quality is already well-calibrated
- GATK documentation acknowledges < 0.5% improvement in sensitivity on current platforms
- Tutorial BAMs are already well-characterized GIAB samples

In [ ]:
# Simulate BQSR before/after base quality distribution
np.random.seed(3)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# ── Left: Before BQSR (miscalibrated) ────────────────────────────────────────
reported_quality = np.repeat([10,15,20,25,28,30,35,40], [500,1000,3000,5000,8000,15000,12000,5000])
# Actual errors are slightly higher than reported (miscalibration)
# Each 'reported Q' bucket has a spread of actual error rates
actual_before = reported_quality + np.random.normal(-0.8, 1.2, len(reported_quality))
actual_before = np.clip(actual_before, 5, 42)

axes[0].scatter(reported_quality + np.random.normal(0,0.3,len(reported_quality)),
                actual_before, alpha=0.005, s=1, color='red')
# Show mean per reported Q
for q in [10,15,20,25,28,30,35,40]:
    mask = np.abs(reported_quality - q) < 1
    axes[0].scatter(q, actual_before[mask].mean(), color='darkred', s=40, zorder=5)
axes[0].plot([0, 42], [0, 42], 'k--', linewidth=1.5, label='Perfect calibration')
axes[0].set_xlabel('Reported Quality Score')
axes[0].set_ylabel('Actual Quality Score (observed error rate)')
axes[0].set_title('Before BQSR\nReported Q > Actual Q (over-confident)')
axes[0].set_xlim(0, 42); axes[0].set_ylim(0, 42)
axes[0].legend(fontsize=9)

# ── Middle: After BQSR (calibrated) ──────────────────────────────────────────
actual_after = reported_quality + np.random.normal(-0.05, 0.3, len(reported_quality))
actual_after = np.clip(actual_after, 5, 42)

axes[1].scatter(reported_quality + np.random.normal(0,0.3,len(reported_quality)),
                actual_after, alpha=0.005, s=1, color='green')
for q in [10,15,20,25,28,30,35,40]:
    mask = np.abs(reported_quality - q) < 1
    axes[1].scatter(q, actual_after[mask].mean(), color='darkgreen', s=40, zorder=5)
axes[1].plot([0, 42], [0, 42], 'k--', linewidth=1.5, label='Perfect calibration')
axes[1].set_xlabel('Reported Quality Score')
axes[1].set_ylabel('Actual Quality Score')
axes[1].set_title('After BQSR\nReported ≈ Actual (well-calibrated)')
axes[1].set_xlim(0, 42); axes[1].set_ylim(0, 42)
axes[1].legend(fontsize=9)

# ── Right: By cycle (end-of-read quality drop) ────────────────────────────────
cycles = np.arange(1, 151)
q_before = 37 - 0.02*cycles - 0.0005*(cycles**2) + np.random.normal(0,0.3,150)
q_after  = 36.5 - 0.025*cycles + np.random.normal(0,0.15,150)

axes[2].plot(cycles, q_before, color='red',   linewidth=2, label='Before BQSR')
axes[2].plot(cycles, q_after,  color='green', linewidth=2, label='After BQSR')
axes[2].axhline(30, color='gray', linestyle='--', linewidth=1, label='Q30 threshold')
axes[2].set_xlabel('Read Cycle (Position in Read)')
axes[2].set_ylabel('Mean Quality Score')
axes[2].set_title('AnalyzeCovariates: Quality by Cycle\n(BQSR corrects end-of-read bias)')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.savefig('../results/docs/bqsr_explainer.png', bbox_inches='tight')
plt.show()

print('EXPECTED RECAL TABLE (excerpt):')
print('ReadGroup  QualityScore  CycleCovariate  ContextCovariate  EventType  EmpiricalQ  Observations')
print('sample1    30            1               GA                M          29.87       1250000')
print('sample1    30            145             GA                M          27.12       890000   ← end of read, lower actual Q')
print('sample1    30            75              TG                M          30.23       1100000  ← middle of read, well calibrated')

print()
print('=== PRODUCTION COMMAND ===')
print('''
# Step 1: Build recalibration table (~2 hours, ~16 GB RAM)
gatk --java-options "-Xmx16g" BaseRecalibrator \
    -I results/preprocessing/tumor.markdup.bam \
    -R data/ref/Homo_sapiens_assembly38.fasta \
    --known-sites data/resources/dbsnp_146.hg38.vcf.gz \
    --known-sites data/resources/Mills_and_1000G_gold_standard.indels.hg38.vcf.gz \
    --known-sites data/resources/1000G_phase1.snps.high_confidence.hg38.vcf.gz \
    -O results/preprocessing/tumor.recal.table

# Step 2: Apply recalibration
gatk --java-options "-Xmx16g" ApplyBQSR \
    -I results/preprocessing/tumor.markdup.bam \
    -R data/ref/Homo_sapiens_assembly38.fasta \
    --bqsr-recal-file results/preprocessing/tumor.recal.table \
    -O results/preprocessing/tumor.bqsr.bam

# Step 3: Visualize calibration improvement
gatk AnalyzeCovariates \
    -before results/preprocessing/tumor.recal.table \
    -after  results/preprocessing/tumor.post_recal.table \
    -plots  results/preprocessing/bqsr_covariates.pdf
''')

---
## Step 5: VQSR — Variant Quality Score Recalibration

### Concept
VQSR is the GATK-recommended method for filtering germline variant callsets with > 1,000 samples
or genome-wide data. It trains a Gaussian mixture model (GMM) on variant annotation features
to distinguish true variants from artifacts.

**Core idea**: True variants and artifacts have different distributions in multidimensional
annotation space (QD, MQ, FS, SOR, etc.). VQSR learns these distributions from validated
truth resources and assigns each variant a log-odds score (VQSLOD).

### Truth resources used for training
| Resource | Used as | Prior | Content |
|----------|---------|-------|--------|
| HapMap 3.3 | truth=true, training=true | 15 | High-confidence SNPs in known populations |
| 1000G Omni 2.5 | truth=true, training=true | 12 | Genotyped variants across 1092 individuals |
| 1000G phase1 SNPs | training=true | 10 | High-confidence common SNPs |
| dbSNP | known=true | 2 | All known variants (lower confidence) |

### Why VQSR fails on small callsets
The GMM needs enough variants to estimate the multivariate Gaussian parameters reliably.
With < 10,000 SNPs (our chr20 tutorial callset has ~5,000–8,000), the model fails to converge
or produces nonsensical VQSLOD scores worse than hard filters.

**GATK minimum requirements:**
- SNP VQSR: ≥ 10,000 SNPs in the callset
- Indel VQSR: ≥ 30 WGS samples (or large exome cohort)

### Ti/Tv ratio — the key quality metric
Transitions (Ti): A↔G and C↔T (chemical similarity — more common)
Transversions (Tv): A↔C, A↔T, G↔C, G↔T (less common)

Expected Ti/Tv ratio:
- WGS: ~2.0–2.1
- WES (coding): ~2.6–3.5 (coding regions are enriched for missense → higher Ti)
- Unfiltered callset: < 1.8 (lots of random Tv artifacts from errors)
- Over-filtered callset: > 3.5 (lost real Tv variants)

After VQSR at 99.5% sensitivity tranche, Ti/Tv should approach the expected WGS value.

In [ ]:
# Simulate VQSR tranches plot and Ti/Tv by tranche
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: Tranches plot (sensitivity vs. % novel)
tranches    = [90.0, 92.0, 94.0, 95.0, 96.0, 97.0, 98.0, 99.0, 99.5, 100.0]
# As we increase sensitivity (accept more variants), we accept more novel (potentially false) variants
novel_pct   = [0.5,  0.8,  1.2,  1.6,  2.3,  3.8,  6.5,  10.2, 14.8, 22.1]
titv_by_tranche = [2.52, 2.49, 2.46, 2.43, 2.38, 2.28, 2.15, 2.07, 2.02, 1.85]

ax1 = axes[0]
ax2 = ax1.twinx()
line1 = ax1.plot(tranches, novel_pct, 'o-', color='red',   linewidth=2, label='% Novel variants (FP proxy)')
line2 = ax2.plot(tranches, titv_by_tranche, 's--', color='blue', linewidth=2, label='Ti/Tv ratio')
ax1.axvline(99.5, color='green', linestyle=':', linewidth=2, label='Recommended 99.5% tranche')
ax1.set_xlabel('Sensitivity Tranche (%)')
ax1.set_ylabel('% Novel Variants (potential FP)', color='red')
ax2.set_ylabel('Ti/Tv Ratio', color='blue')
ax2.axhline(2.1, color='blue', linestyle=':', alpha=0.5, label='Expected WGS Ti/Tv (2.0-2.1)')
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left', fontsize=9)
ax1.set_title('VQSR Tranches Plot\n(Sensitivity vs. Specificity Tradeoff)')

# ── Right: Ti/Tv before/after filtration ──────────────────────────────────────
categories = ['Raw\n(unfiltered)', 'Hard filters\n(this pipeline)', 'VQSR 99%\ntranche', 'VQSR 99.5%\ntranche']
titv_values = [1.82, 2.05, 2.08, 2.02]
colors_bar = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4']

axes[1].bar(categories, titv_values, color=colors_bar, edgecolor='white', alpha=0.85)
axes[1].axhline(2.0, color='green', linestyle='--', linewidth=1.5, label='Expected WGS min (2.0)')
axes[1].axhline(2.1, color='blue',  linestyle='--', linewidth=1.5, label='Expected WGS max (2.1)')
axes[1].fill_between([-0.5, 3.5], 2.0, 2.1, alpha=0.1, color='green')
axes[1].set_ylabel('Ti/Tv Ratio')
axes[1].set_title('Ti/Tv Ratio: Raw vs. Filtered Callsets\n(Higher = fewer Tv artifacts)')
axes[1].legend(fontsize=9)
axes[1].set_ylim(1.5, 2.3)

for i, (cat, val) in enumerate(zip(categories, titv_values)):
    axes[1].text(i, val + 0.01, f'{val:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/docs/vqsr_explainer.png', bbox_inches='tight')
plt.show()

print('KEY TAKEAWAY:')
print('  Hard filters achieve Ti/Tv ~2.05 — acceptable for small callsets')
print('  VQSR achieves Ti/Tv ~2.02-2.08 — slightly better, but requires large callset')
print('  For our chr20 tutorial data, hard filters are the correct choice')

print()
print('VQSR FAILURE ON SMALL CALLSETS:')
print('  ERROR: Failed to converge after 150 iterations')
print('  Cause: Not enough variants to estimate Gaussian parameters')
print('  Minimum: ~10,000 SNPs for SNP model; ~30 WGS samples for indel model')

print()
print('=== PRODUCTION COMMAND (requires WGS or large cohort) ===')
print('''
# SNP VQSR
gatk --java-options "-Xmx24g" VariantRecalibrator \
    -R data/ref/Homo_sapiens_assembly38.fasta \
    -V results/germline/genotyped/raw_variants.vcf.gz \
    --resource:hapmap,known=false,training=true,truth=true,prior=15  resources/hapmap_3.3.hg38.vcf.gz \
    --resource:omni,known=false,training=true,truth=false,prior=12   resources/1000G_omni2.5.hg38.vcf.gz \
    --resource:1000G,known=false,training=true,truth=false,prior=10  resources/1000G_phase1.snps.hg38.vcf.gz \
    --resource:dbsnp,known=true,training=false,truth=false,prior=2   resources/dbsnp_146.hg38.vcf.gz \
    -an QD -an MQ -an MQRankSum -an ReadPosRankSum -an FS -an SOR \
    -mode SNP \
    -O results/germline/filtered/snps.recal \
    --tranches-file results/germline/filtered/snps.tranches \
    --rscript-file results/germline/filtered/snps_plots.R

# Apply SNP VQSR at 99.5% sensitivity
gatk ApplyVQSR \
    -R data/ref/Homo_sapiens_assembly38.fasta \
    -V results/germline/genotyped/raw_variants.vcf.gz \
    --recal-file results/germline/filtered/snps.recal \
    --tranches-file results/germline/filtered/snps.tranches \
    --truth-sensitivity-filter-level 99.5 \
    --create-output-variant-index true \
    -mode SNP \
    -O results/germline/filtered/snp_vqsr.vcf.gz
''')

---
## Summary: When to use each approach

| Scenario | Read QC | Adapter Trim | BQSR | Variant Filtering |
|----------|---------|-------------|------|------------------|
| **This tutorial** (pre-aligned BAMs) | Skipped | Skipped | Skipped | Hard filters |
| **Small gene panel** (< 100 genes) | Yes | Yes | Optional | Hard filters |
| **Clinical exome** (single sample) | Yes | Yes | Yes | Hard filters |
| **Research WGS** (single sample) | Yes | Yes | Yes | Hard filters |
| **Cohort WGS** (30–1000 samples) | Yes | Yes | Yes | VQSR |
| **Population WGS** (gnomAD scale) | Yes | Yes | Yes | VQSR + ML models |

### Memory requirements reference
| Step | RAM needed | CPU hours (30× WGS sample) |
|------|-----------|---------------------------|
| FastQC | < 500 MB | 0.5h |
| fastp | < 300 MB | 0.3h |
| BWA-MEM | 8 GB | 4h (16 threads) |
| BQSR | 16–32 GB | 2h |
| Mutect2 / HaplotypeCaller | 8–16 GB | 4–8h |
| VQSR | 24 GB | 2h |
| Annotation (API-based) | < 2 GB | 0.5–2h (API latency) |